# Create Riksbankens Jubileumsfond (RJ) Awards (GRANT PATTERN, method-5 static HTML)

RJ grants from the foundation's own Drupal-published archive at rj.se. Sweden's leading humanities and social-sciences research funder (~SEK 600-700M/year since 1965).

**Prerequisites:**
- Run `scripts/local/rj_jubileumsfond_to_s3.py` first.

**Data source:** Drupal sitemap at rj.se/sitemap.xml. 4,083 total URLs; filtered to `/en/grants/{YEAR}/{slug}/` → **1,676 English grant pages**. Swedish-only grants (no EN counterpart) are out of scope for this MVP.

Each EN grant page has a `<div class="contentBox grantInfoBox">` with span+div pairs:
- Grant administrator → institution
- Reference number → native award ID (e.g., "RMP20-0015")
- Amount → SEK with commas (e.g., "SEK 976,000")
- Funding → program name (e.g., "RJ Flexit")
- Subject → topic
- Year → decision year

Plus `<h1>` title and PI name (first short text element after h1).

**S3 location:** `s3a://openalex-ingest/awards/rj/rj_grants.parquet`

**Awarding body:**
- funder_id: 4320322659
- display_name: "Riksbankens Jubileumsfond"
- country: SE
- ROR: none
- DOI: 10.13039/501100004472

**Coverage (smoke test 10 grants, 2026-05-23):**
- 100% title / slug / reference_number / grant_administrator / amount / currency / award_year / funding_program / subject
- 10% description (most RJ pages don't render a long abstract; the structured fields carry the substance)

**Amount/currency:** POPULATED from the structured "Amount" field. Currency = SEK (RJ funds Swedish-currency only). NOT a §6.7 waiver — RJ publishes per-grant amounts structurally on every page.

**Provenance:** `rj_jubileumsfond_grants`.

**Priority:** 118.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rj_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rj/rj_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.rj_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.rj_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

RJ publishes amounts structurally — expect 95%+ pct_amount.

In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'rj_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'rj_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT title, slug, reference_number, grant_administrator, amount, currency, award_year, funding_program, subject, pi_raw
FROM openalex.awards.rj_raw LIMIT 5;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    SUM(CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS rows_parseable,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amt,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amt,
    ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_amt,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)) / 1e9, 2) AS total_sek_billions
FROM openalex.awards.rj_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.rj_raw;


In [ ]:
%sql
SELECT award_year, COUNT(*) as grants
FROM openalex.awards.rj_raw
WHERE award_year IS NOT NULL
GROUP BY award_year ORDER BY award_year DESC LIMIT 30;


In [ ]:
%sql
SELECT funding_program, COUNT(*) as grants, ROUND(SUM(TRY_CAST(amount AS DOUBLE))/1e6, 1) AS total_sek_millions
FROM openalex.awards.rj_raw
GROUP BY funding_program ORDER BY grants DESC LIMIT 20;


In [ ]:
%sql
SELECT grant_administrator, COUNT(*) as cnt
FROM openalex.awards.rj_raw
GROUP BY grant_administrator ORDER BY cnt DESC LIMIT 15;


## Step 1.6: Fail-fast — Verify the RJ Funder Row Exists

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320322659;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rj_awards
USING delta
AS
WITH
rj_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322659
),
raw_prepared AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(award_year AS INT) AS parsed_year,
        TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.rj_raw
    WHERE title IS NOT NULL
      AND TRIM(title) != ''
      AND funder_award_id IS NOT NULL
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.title as display_name,
        g.description as description,

        f.funder_id,
        g.funder_award_id,

        g.parsed_amount as amount,
        g.currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'research' as funding_type,
        COALESCE(NULLIF(TRIM(g.funding_program), ''), 'RJ Grant') as funder_scheme,

        'rj_jubileumsfond_grants' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        -- PI name NULL pending scraper fix (oxjobs #267 — backlog/fix-rj-scraper-pi-extraction).
        -- The scraper's pi_raw field captures the wrong DOM node: "Final report" link text or
        -- grant_administrator string, never the real PI. Affiliation kept (grant_administrator
        -- is reliable) — HHMI / Helmsley / Mott org-level precedent.
        CASE
            WHEN g.grant_administrator IS NULL THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                CAST(NULL AS STRING) as given_name,
                CAST(NULL AS STRING) as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    NULLIF(TRIM(g.grant_administrator), '') as name,
                    'SE' as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN rj_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 118

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rj_jubileumsfond_grants' AND priority = 118;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    118 as priority
FROM openalex.awards.rj_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.rj_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.rj_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_start_date,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) as pct_pi
FROM openalex.awards.rj_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, funder_scheme,
       amount, currency,
       lead_investigator.given_name AS pi_given,
       lead_investigator.family_name AS pi_family,
       lead_investigator.affiliation.name AS pi_affiliation
FROM openalex.awards.rj_awards LIMIT 10;


In [ ]:
%sql
-- §6.7 amount check. Expected: ~95%+ pct_amount, SEK only.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt,
    ROUND(SUM(amount) / 1e9, 2) AS total_sek_billions
FROM openalex.awards.rj_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.rj_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.rj_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 30;


In [ ]:
%sql
SELECT COUNT(*) AS total, COUNT(DISTINCT funder_award_id) AS distinct_ids
FROM openalex.awards.rj_awards;
